# 4.1 · Your first server with the Python SDK

**用 Python SDK 写第一个服务器**

> **After this lesson you can:** Build, run and connect a real MCP server with the Python SDK — tools from type hints, lifespan-loaded state, Context progress, stdio and Streamable HTTP, and a working Claude Desktop config.

Official reading: [Build an MCP server](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-server) · [SDKs](https://modelcontextprotocol.io/docs/2026-07-28/sdk)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. A fake National Weather Service (no network)
The official tutorial calls `api.weather.gov`. To stay offline and deterministic, we save a tiny NWS-shaped dataset to a temp folder. The server loads it **once** through a lifespan. Its path is passed as an **absolute** path in an environment variable, exactly as you would do in `claude_desktop_config.json`.

In [ ]:
import json
import os
import sys
import tempfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)
workdir = Path(tempfile.mkdtemp(prefix="weather-"))
server_path = workdir / "weather.py"
data_path = workdir / "nws_sample.json"

nws = {
    "alerts": {
        "CA": [{"event": "Excessive Heat Warning", "areaDesc": "Sacramento Valley", "severity": "Severe",
                "description": "Dangerously hot conditions with temperatures up to 110.",
                "instruction": "Drink plenty of fluids and stay in an air-conditioned room."}],
        "TX": [{"event": "Flood Watch", "areaDesc": "Harris County", "severity": "Moderate",
                "description": "Heavy rain may cause flooding of low-lying areas.",
                "instruction": "Turn around, don't drown."},
               {"event": "Heat Advisory", "areaDesc": "Bexar County", "severity": "Minor",
                "description": "Heat index values up to 108 expected."}],
        "NY": [],
    },
    "points": {  # "lat,lon" rounded to 2 decimals -> forecast periods
        "38.58,-121.49": {"city": "Sacramento", "periods": [
            {"name": f"Period {i}", "temperature": 100 - 3 * i, "temperatureUnit": "F", "windSpeed": "5 mph",
             "windDirection": "SW", "detailedForecast": "Sunny and hot." if i % 2 == 0 else "Clear."}
            for i in range(1, 9)]},
        "40.71,-74.01": {"city": "New York", "periods": [
            {"name": "Tonight", "temperature": 61, "temperatureUnit": "F", "windSpeed": "10 mph",
             "windDirection": "NE", "detailedForecast": "Mostly cloudy."}]},
    },
}
data_path.write_text(json.dumps(nws, indent=1))
os.environ["WEATHER_DATA"] = str(data_path)          # used when we load the server in-process
print("dataset:", data_path, f"({data_path.stat().st_size} bytes)")

### 2. The server file
This is the tutorial's `weather.py` with three upgrades: a **lifespan** that loads the dataset once, `Annotated` field descriptions and ranges, and `Context` progress reporting. All logging goes to **stderr**. The `NOISY` switch adds two deliberate `print()` bugs that we trigger in section 6.

In [ ]:
%%writefile {server_path}
"""Offline weather MCP server, shaped like the official tutorial."""
import json
import logging
import os
import sys
from contextlib import asynccontextmanager
from dataclasses import dataclass
from typing import Annotated

from pydantic import Field

from mcp.server import MCPServer
from mcp.server.mcpserver import Context

logging.basicConfig(level=logging.INFO, stream=sys.stderr,           # stderr, never stdout
                    format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("weather")

if os.environ.get("NOISY"):
    print("Weather server starting...", flush=True)   # BUG on stdio: not a JSON-RPC message


@dataclass
class WeatherData:
    alerts: dict
    points: dict


@asynccontextmanager
async def lifespan(server: MCPServer):
    path = os.environ["WEATHER_DATA"]       # absolute path, set by the host config's "env"
    logger.info("lifespan: loading %s", path)
    with open(path) as f:
        raw = json.load(f)
    yield WeatherData(alerts=raw["alerts"], points=raw["points"])
    logger.info("lifespan: shutting down")


mcp = MCPServer("weather", lifespan=lifespan)


def format_alert(a: dict) -> str:
    return (f"Event: {a.get('event', 'Unknown')}\nArea: {a.get('areaDesc', 'Unknown')}\n"
            f"Severity: {a.get('severity', 'Unknown')}\n"
            f"Description: {a.get('description', 'No description available')}\n"
            f"Instructions: {a.get('instruction', 'No specific instructions provided')}")


@mcp.tool()
async def get_alerts(state: Annotated[str, Field(description="Two-letter US state code, e.g. CA or NY")],
                     ctx: Context) -> str:
    """Get active weather alerts for a US state."""
    if os.environ.get("NOISY"):
        print(f"looking up alerts for {state}")    # BUG: where does this end up? (section 6)
    data: WeatherData = ctx.request_context.lifespan_context
    logger.info("get_alerts state=%s", state)
    features = data.alerts.get(state.upper())
    if features is None:
        return f"Unable to fetch alerts for {state!r}. Use a two-letter US state code such as CA."
    if not features:
        return "No active alerts for this state."
    return "\n---\n".join(format_alert(a) for a in features)


@mcp.tool()
async def get_forecast(latitude: Annotated[float, Field(description="Latitude of the location", ge=-90, le=90)],
                       longitude: Annotated[float, Field(description="Longitude of the location", ge=-180, le=180)],
                       ctx: Context) -> str:
    """Get the weather forecast (next 5 periods) for a US location."""
    data: WeatherData = ctx.request_context.lifespan_context
    await ctx.report_progress(0, 2, "resolving grid point")
    grid = data.points.get(f"{latitude:.2f},{longitude:.2f}")
    if grid is None:
        return "Unable to fetch forecast data for this location (only US locations are supported)."
    await ctx.report_progress(1, 2, "fetching forecast periods")
    periods = [f"{p['name']}:\nTemperature: {p['temperature']}°{p['temperatureUnit']}\n"
               f"Wind: {p['windSpeed']} {p['windDirection']}\nForecast: {p['detailedForecast']}"
               for p in grid["periods"][:5]]                                # only the next 5 periods
    await ctx.report_progress(2, 2, "done")
    return "\n---\n".join(periods)


if __name__ == "__main__":
    if len(sys.argv) == 3 and sys.argv[1] == "http":
        mcp.run(transport="streamable-http", host="127.0.0.1", port=int(sys.argv[2]))
    else:
        mcp.run(transport="stdio")

### 3. What the SDK generated from your type hints
Load the file as a module and connect in-process. Look at the wire form of `tools/list`. The docstring became `description`. `Annotated` metadata became property `description`, `minimum` and `maximum`. The `ctx` parameter is absent. The `-> str` return produced an `outputSchema` with a `result` field.

In [ ]:
import importlib.util
import logging

from mcp import Client

spec = importlib.util.spec_from_file_location("weather_server", server_path)
weather = importlib.util.module_from_spec(spec)
spec.loader.exec_module(weather)
logging.getLogger("weather").setLevel(logging.WARNING)       # keep the notebook output tidy

async with Client(weather.mcp) as client:
    listed = await client.list_tools()
    wire = listed.model_dump(by_alias=True, exclude_none=True, mode="json")
    print(json.dumps(wire["tools"][1], indent=2))
    print("\nall tools:", [t.name for t in listed.tools])

### 4. Call the tools, with progress
`get_forecast` calls `ctx.report_progress`. The client only receives progress if it asks: passing `progress_callback` makes the Python client attach a `progressToken` to that one request.

In [ ]:
progress_seen = []


async def on_progress(progress, total, message):
    progress_seen.append(f"{progress}/{total} {message}")


async with Client(weather.mcp) as client:
    r = await client.call_tool("get_forecast", {"latitude": 38.58, "longitude": -121.49},
                               progress_callback=on_progress)
    print(r.content[0].text.split("\n---\n")[0], "\n...")
    print("periods returned:", r.content[0].text.count("Temperature"), "(capped at 5 of 8)")
    print("progress notifications:", progress_seen)

    r = await client.call_tool("get_alerts", {"state": "TX"})
    print("\nTX alerts:\n" + r.content[0].text)

    bad = await client.call_tool("get_forecast", {"latitude": 123, "longitude": 0})
    print("\nlatitude=123 -> isError:", bad.is_error, "|", bad.content[0].text.splitlines()[1:3])

The last call never reached your function. Pydantic enforced `le=90` and the SDK turned the failure into an `isError: true` result the model can read and correct.

### 5. Run it for real over stdio, with a host-style log file
A host like Claude Desktop **spawns** the server and writes its stderr to `mcp-server-weather.log`. We do the same: `stdio_client(params, errlog=file)` is a transport we can hand to `Client`. Note the `env` (the host's `"env"` key) and the absolute path to the script.

In [ ]:
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

params = StdioServerParameters(command=sys.executable, args=[str(server_path)],
                               env={"WEATHER_DATA": str(data_path)})
log_file = workdir / "mcp-server-weather.log"

with open(log_file, "w") as errlog:
    async with Client(stdio_client(params, errlog=errlog)) as client:
        print("connected over stdio, protocol", client.protocol_version, "server", client.server_info.name)
        for state in ["CA", "NY"]:
            r = await client.call_tool("get_alerts", {"state": state})
            print(f"{state}:", r.content[0].text.splitlines()[0])

print("\n--- mcp-server-weather.log (the server's stderr) ---")
print(log_file.read_text())

The log shows the lifespan loading the data **once**, before the tool calls, and shutting down when the client closed stdin.

### 6. Break it: a stray `print()` on stdout
Now set `NOISY=1` and talk to the server by hand. The first thing the server writes to stdout is not JSON. A strict client that does `json.loads(line)` on every line fails immediately.

In [ ]:
import subprocess

META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "strict-client", "version": "0"}}

proc = subprocess.Popen([sys.executable, str(server_path)], text=True,
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                        env={**os.environ, "NOISY": "1", "WEATHER_DATA": str(data_path)})
proc.stdin.write(json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tools/call",
                             "params": {"name": "get_alerts", "arguments": {"state": "NY"}, "_meta": META}}) + "\n")
proc.stdin.flush()

for n in range(1, 4):                       # read until the response with id 1 arrives
    line = proc.stdout.readline()
    try:
        msg = json.loads(line)
    except json.JSONDecodeError as exc:
        print(f"line {n}: NOT JSON -> {line.strip()!r}  ({exc.msg}) <- a strict client dies here")
        continue
    print(f"line {n}: valid JSON-RPC, id={msg.get('id')} -> {msg['result']['content'][0]['text']!r}")
    if msg.get("id") == 1:
        break

proc.stdin.close()                          # the client hangs up -> the server shuts down
proc.wait(timeout=10)
late_stdout, stderr_text = proc.stdout.read(), proc.stderr.read()
print("\nwritten to stdout AFTER the last response:", repr(late_stdout))
print("in-tool print found in stderr?", "looking up alerts for NY" in stderr_text)

Read both results carefully:

- The **startup** `print()` ran before `mcp.run()`, so it polluted the protocol channel.
- The **in-tool** `print()` (not flushed) sat in Python's stdout buffer and was written to the *protocol stream* at shutdown. SDK 2.x points fd 1 at stderr while serving, but it restores the real stdout on exit, and the buffer is flushed after that. Add `flush=True` to that print and re-run: it lands in stderr instead. Either way the result depends on buffering, which is exactly why you never print on stdio.
- The bug uses `flush=True` so it shows up deterministically. Without it, Python block-buffers stdout when stdout is a pipe, and the stray text appears at some unpredictable later moment, possibly in the middle of a message. That is even harder to debug.

The Python SDK's own client is lenient (it logs a parse error and skips the line). Other hosts are not obliged to be. The rule stays: on stdio, log to stderr.

### 7. The same server over Streamable HTTP
No code changes: run the file with `http PORT` and connect to `http://127.0.0.1:PORT/mcp`. We keep `NOISY=1` on purpose. Over HTTP, stdout is just a console, so the prints are harmless.

In [ ]:
import socket
import time

for noisy in ("httpx", "httpx2"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]

http_log = open(workdir / "http-server.log", "w")
http_proc = subprocess.Popen([sys.executable, str(server_path), "http", str(port)],
                             env={**os.environ, "NOISY": "1", "WEATHER_DATA": str(data_path)},
                             stdout=http_log, stderr=subprocess.STDOUT)
for _ in range(100):                                     # wait until uvicorn accepts connections
    try:
        socket.create_connection(("127.0.0.1", port), timeout=0.2).close()
        break
    except OSError:
        time.sleep(0.1)

try:
    url = f"http://127.0.0.1:{port}/mcp"
    for attempt in range(2):                             # two independent clients, one server process
        async with Client(url) as client:
            r = await client.call_tool("get_alerts", {"state": "CA"})
            print(f"client {attempt + 1}: {client.protocol_version} ->", r.content[0].text.splitlines()[0])
finally:
    http_proc.terminate()
    http_proc.wait(timeout=10)
    http_log.close()

print("\n--- server console (stdout + stderr) ---")
print("\n".join(l for l in (workdir / "http-server.log").read_text().splitlines() if "POST" not in l))

The console shows the noisy `print`, uvicorn's own messages and one lifespan load for the whole process, shared by both clients.

### 8. Generate the Claude Desktop config
A small helper writes the `mcpServers` entry the way the tutorial recommends (`uv --directory <abs> run weather.py`). It also writes the direct-interpreter variant, and **lints** the result for the classic mistakes: relative paths, a bare command a GUI app may not find, and missing `env`.

In [ ]:
import shutil


def desktop_entry(server_file: Path, use_uv: bool = True) -> dict:
    env = {"WEATHER_DATA": str(data_path)}
    if use_uv:
        return {"command": shutil.which("uv") or "uv",
                "args": ["--directory", str(server_file.parent), "run", server_file.name], "env": env}
    return {"command": sys.executable, "args": [str(server_file)], "env": env}


def lint_entry(name: str, entry: dict) -> list[str]:
    problems = []
    if not os.path.isabs(entry["command"]):
        problems.append(f"{name}: command {entry['command']!r} is not absolute; GUI hosts may not see your PATH")
    for arg in entry.get("args", []):
        if ("/" in arg or "\\" in arg) and not os.path.isabs(arg):
            problems.append(f"{name}: relative path {arg!r}; the host's working directory may be '/'")
    for key, value in entry.get("env", {}).items():
        if value.endswith(".json") and not os.path.isabs(value):
            problems.append(f"{name}: env {key} is a relative path")
    return problems


config = {"mcpServers": {"weather": desktop_entry(server_path, use_uv=True),
                         "weather-python": desktop_entry(server_path, use_uv=False),
                         "broken": {"command": "uv", "args": ["--directory", "./weather", "run", "weather.py"]}}}
print(json.dumps(config, indent=2))
print("\nlint:")
for name, entry in config["mcpServers"].items():
    print(" ", name, "->", lint_entry(name, entry) or "ok")
print("\nmacOS file: ~/Library/Application Support/Claude/claude_desktop_config.json  (then fully quit Claude)")

### 9. Under the hood: the six steps, with a scripted "Claude"
The tutorial's "what's happening under the hood" list, as code. A deterministic stand-in for the model picks a tool from the advertised descriptions; the host runs it through MCP and feeds the result back. If `ANTHROPIC_API_KEY` is set, the real model does steps 2 and 5.

In [ ]:
async def fake_model(question: str, tools) -> dict:
    """Step 2: choose a tool by reading names/descriptions (a stand-in for the LLM)."""
    q = question.lower()
    if "alert" in q:
        state = [w.strip("?.") for w in question.split() if w.strip("?.").isupper() and len(w.strip("?.")) == 2][0]
        return {"name": "get_alerts", "arguments": {"state": state}}
    return {"name": "get_forecast", "arguments": {"latitude": 38.58, "longitude": -121.49}}


question = "Are there any weather alerts in TX?"
async with Client(weather.mcp) as client:
    tools = (await client.list_tools()).tools
    print("1. host sends the question + tool list to the model:", question)
    choice = await fake_model(question, tools)
    print("2. model chooses:", choice)
    result = await client.call_tool(choice["name"], choice["arguments"])
    print("3. host executes it through MCP -> isError:", result.is_error)
    print("4. result goes back to the model:", result.content[0].text.count("Event:"), "alerts")
    events = [l.split(": ", 1)[1] for l in result.content[0].text.splitlines() if l.startswith("Event:")]
    print("5./6. answer shown to the user: Texas has", len(events), "active alerts:", ", ".join(events))

if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    llm = anthropic.Anthropic()
    async with Client(weather.mcp) as client:
        tool_defs = [{"name": t.name, "description": t.description, "input_schema": t.input_schema}
                     for t in (await client.list_tools()).tools]
        messages = [{"role": "user", "content": "What's the weather in Sacramento (38.58, -121.49)?"}]
        response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=tool_defs, messages=messages)
        while response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type == "tool_use":
                    r = await client.call_tool(block.name, block.input)
                    results.append({"type": "tool_result", "tool_use_id": block.id,
                                    "content": r.content[0].text, "is_error": bool(r.is_error)})
            messages.append({"role": "user", "content": results})
            response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=tool_defs, messages=messages)
        print("\nClaude:", "".join(b.text for b in response.content if b.type == "text"))
else:
    print("\n(skipped real model: set ANTHROPIC_API_KEY to let Claude do steps 2 and 5)")

### 10. The Context object beyond progress
A small companion server shows the rest of `Context`. It reads one of its own resources from inside a tool, inspects request metadata, and uses the **deprecated** protocol logging. Protocol logging only reaches a 2026-07-28 client that opted in with `log_level`.

In [ ]:
from mcp.server import MCPServer
from mcp.server.mcpserver import Context

ctx_demo = MCPServer("ctx-demo")


@ctx_demo.resource("config://units")
def units() -> str:
    return "imperial"


@ctx_demo.tool()
async def describe_request(ctx: Context) -> dict[str, str]:
    """Report what the Context object knows about this request."""
    units_text = list(await ctx.read_resource("config://units"))[0].content
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        await ctx.info("describe_request called")          # deprecated notifications/message
    return {"request_id": ctx.request_id, "protocol": ctx.protocol_version or "", "units": units_text}


for level in [None, "info"]:
    received = []

    async def on_log(params):
        received.append(f"[{params.level}] {params.data}")

    async with Client(ctx_demo, logging_callback=on_log, log_level=level) as client:
        r = await client.call_tool("describe_request", {})
        print(f"log_level={level!r:7} -> structured={r.structured_content} | log messages received={received}")

### Your turn
1. Add a third tool, `get_station_count(state: Literal["CA", "TX", "NY"]) -> int`, to the server file. Re-run sections 2–3 and find the `enum` in its `inputSchema` and the integer `outputSchema`.
2. Move the `NOISY` startup `print` to `sys.stderr` (`print(..., file=sys.stderr)`) and re-run section 6. Which line is now the first one on stdout?
3. Make the lifespan fail (point `WEATHER_DATA` at a missing file in section 5's `env`). What does the client raise, and where in `mcp-server-weather.log` is the real cause?
4. Change `desktop_entry` to also emit the Windows form of the path (double backslashes) and extend `lint_entry` to catch a missing `env` entry for `WEATHER_DATA`.